# Case Study Report Assignment 1: Asking Questions About Data

This notebook analyzes academic performance records for engineering students. It answers the four required questions about parental education, socioeconomic stratum, gender, and recommendations.

## 1. Load and clean the data

Place `data_academic_performance.xlsx` in the same folder as this notebook when running locally, or upload it to Google Colab. The workbook also contains variable-definition sheets.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

FILE = 'data_academic_performance.xlsx'
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)
if not os.path.exists(FILE):
    from google.colab import files
    uploaded = files.upload()
    FILE = next(iter(uploaded))

workbook = pd.ExcelFile(FILE)
raw = pd.read_excel(workbook, sheet_name='SABER11_SABERPRO')
df = raw.drop(columns=['Unnamed: 9'], errors='ignore').copy()
df['STRATUM'] = df['STRATUM'].replace({0: 'Stratum 0/unknown'})
print('Workbook sheets:', workbook.sheet_names)
print('Raw worksheet shape:', raw.shape)
print('Rows and columns after cleaning:', df.shape)
print('Empty columns removed:', [c for c in raw.columns if raw[c].isna().all()])
print('Unknown stratum records:', int((raw['STRATUM'] == 0).sum()))
df.head()

## 2. Variable preparation

Parental education labels are trimmed and mapped to an ordered scale. The values `0` and `Not sure` are treated as missing for ordinal analyses. `G_SC` is the primary outcome; `PRO_MEAN` is a descriptive mean of the five core professional subscores.

In [ ]:
edu_order = {
    'Ninguno': 0, 'Incomplete primary': 1, 'Complete primary': 2,
    'Incomplete Secundary': 3, 'Complete Secundary': 4,
    'Incomplete technical or technological': 5,
    'Complete technique or technology': 6,
    'Incomplete Professional Education': 7,
    'Complete professional education': 8, 'Postgraduate education': 9,
    'Not sure': np.nan, 0: np.nan
}
for parent in ['EDU_FATHER', 'EDU_MOTHER']:
    df[parent] = df[parent].astype(str).str.strip()
    df[parent + '_ORD'] = df[parent].map(edu_order)
df['PARENT_MAX_EDU'] = df[['EDU_FATHER_ORD', 'EDU_MOTHER_ORD']].max(axis=1)
pro_cols = ['QR_PRO', 'CR_PRO', 'CC_PRO', 'ENG_PRO', 'WC_PRO']
df['PRO_MEAN'] = df[pro_cols].mean(axis=1)
num_cols = ['MAT_S11','CR_S11','CC_S11','BIO_S11','ENG_S11','QR_PRO','CR_PRO','CC_PRO','ENG_PRO','WC_PRO','FEP_PRO','G_SC','PERCENTILE','2ND_DECILE','QUARTILE','SEL','SEL_IHE']

df[['G_SC'] + num_cols].describe().T
# Father- and mother-education category summaries used in the report
parent_education_summaries = {}
for parent, label in [('EDU_FATHER', 'Father'), ('EDU_MOTHER', 'Mother')]:
    summary = (
        df.dropna(subset=[parent, 'G_SC'])
          .groupby(parent)['G_SC']
          .agg(['count', 'mean', 'std'])
          .reset_index()
    )
    summary['ci95'] = 1.96 * summary['std'] / np.sqrt(summary['count'])
    parent_education_summaries[label] = summary
    print(f'{label} education summary:')
    display(summary.round(2))
print('Missing values in analyzed numeric columns:', int(df[num_cols].isna().sum().sum()))


## 3. Numerical attributes by gender and stratum

In [ ]:
gender_summary = df.groupby('GENDER')[num_cols].agg(['count','mean','std']).round(2)
stratum_summary = df.groupby('STRATUM')[num_cols].agg(['count','mean','std']).round(2)
display(gender_summary)
display(stratum_summary)

# Gender counts and percentages used in the report
gender_counts = df['GENDER'].value_counts().rename_axis('GENDER').to_frame('count')
gender_counts['percent'] = 100 * gender_counts['count'] / len(df)
display(gender_counts.round(1))


## 4. Question 1 — Does parental education influence outcomes?

In [ ]:
parent_table = df.groupby('PARENT_MAX_EDU', dropna=False)['G_SC'].agg(['count','mean','std']).reset_index()
parent_table['ci95'] = 1.96 * parent_table['std'] / np.sqrt(parent_table['count'])
parent_table['Education level'] = parent_table['PARENT_MAX_EDU'].map({0:'None',1:'Incomplete primary',2:'Complete primary',3:'Incomplete secondary',4:'Complete secondary',5:'Incomplete technical',6:'Complete technical',7:'Incomplete professional',8:'Complete professional',9:'Postgraduate education'})
print('Correlation with G_SC:', df[['PARENT_MAX_EDU','G_SC']].corr().iloc[0,1])
print('ANOVA by father education:', stats.f_oneway(*[g['G_SC'].to_numpy() for _, g in df.dropna(subset=['EDU_FATHER_ORD']).groupby('EDU_FATHER_ORD')]))
print('ANOVA by mother education:', stats.f_oneway(*[g['G_SC'].to_numpy() for _, g in df.dropna(subset=['EDU_MOTHER_ORD']).groupby('EDU_MOTHER_ORD')]))
display(parent_table[parent_table['PARENT_MAX_EDU'].notna()].round(2))
sns.barplot(data=parent_table.dropna(), x='PARENT_MAX_EDU', y='mean', color='#4C91CA')
plt.title('Mean G_SC by highest parent education')
plt.xlabel('Ordered education code (0 = none, 9 = postgraduate)'); plt.ylabel('Mean G_SC'); plt.show(); plt.close()


In [ ]:
model_data = df[['G_SC','PARENT_MAX_EDU','GENDER','STRATUM','SEL','SEL_IHE']].dropna().copy()
model_data['STRATUM_NUM'] = model_data['STRATUM'].str.extract(r'(\d+)')[0].astype(float)
model = smf.ols('G_SC ~ PARENT_MAX_EDU + C(GENDER) + STRATUM_NUM + SEL + SEL_IHE', data=model_data).fit()
print(model.summary())

## 5. Question 2 — Which stratum performs best?

In [ ]:
ordered = [f'Stratum {i}' for i in range(1,7)]
stratum_means = df[df['STRATUM'].isin(ordered)].groupby('STRATUM')['G_SC'].agg(['count','mean','std']).reindex(ordered)
stratum_means['ci95'] = 1.96 * stratum_means['std'] / np.sqrt(stratum_means['count'])
print(stratum_means.round(2))
print('ANOVA for Strata 1-6:', stats.f_oneway(*[g['G_SC'].to_numpy() for _, g in df[df['STRATUM'].isin(ordered)].groupby('STRATUM')]))
stratum_plot = stratum_means.reset_index()
ax = sns.barplot(data=stratum_plot, x='STRATUM', y='mean', order=ordered, color='#3676A8', errorbar=None)
ax.errorbar(
    np.arange(len(stratum_plot)),
    stratum_plot['mean'],
    yerr=stratum_plot['ci95'],
    fmt='none', ecolor='black', elinewidth=2, capsize=4
)
plt.title('Global score by socioeconomic stratum'); plt.ylabel('Mean G_SC (95% CI)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'global_score_by_stratum.png'), dpi=180, bbox_inches='tight')
plt.show()
plt.close()


## 6. Question 3 — How does each gender perform in the global assessment?

In [ ]:
gender_gsc = df.groupby('GENDER')['G_SC'].agg(['count','mean','std'])
print(gender_gsc.round(2))
f = df.loc[df.GENDER=='F','G_SC']; m = df.loc[df.GENDER=='M','G_SC']
print('Welch t-test:', stats.ttest_ind(m, f, equal_var=False))
pooled_sd = np.sqrt(((len(f)-1)*f.var(ddof=1) + (len(m)-1)*m.var(ddof=1)) / (len(f)+len(m)-2))
print('Cohen d (M - F):', (m.mean()-f.mean())/pooled_sd)
subscores = ['G_SC','QR_PRO','CR_PRO','CC_PRO','ENG_PRO','WC_PRO']
sub = df.groupby('GENDER')[subscores].mean().T
display(sub.round(2))
sub.plot(kind='bar', figsize=(10,5)); plt.ylabel('Mean score'); plt.title('Global and professional subscores by gender')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'scores_by_gender.png'), dpi=180, bbox_inches='tight')
plt.show()
plt.close()

## 7. Question 4 — Findings, recommendations, and limitations

In [ ]:
print('Private vs public G_SC:')
print(df.groupby('SCHOOL_NAT')['G_SC'].agg(['count','mean','std']).round(2))
print('Largest stratum mean gap:', stratum_means['mean'].max() - stratum_means['mean'].min())
print('Main findings:')
print('1. Stratum 6 has the highest mean G_SC and Stratum 1 the lowest.')
print('2. Parent education is positively associated with G_SC, but it is not a causal estimate.')
print('3. The overall gender gap is small; component gaps differ by skill.')
print('4. Recommended response: tiered support by stratum and family education, with quantitative and writing supports monitored by gender and program.')
print('Limitations: observational data, no intervention or longitudinal measures, potential selection and omitted-variable bias, and small program groups.')

## References

De La Hoz, Enrique. “Data of Academic Performance Evolution for Engineering Students.” Mendeley Data, V1, 2020, doi:10.17632/83tcx8psxv.1.

Instituto Colombiano para la Evaluación de la Educación (ICFES). Variable definitions in the supplied workbook.